In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "data/",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_data = train_datagen.flow_from_directory(
    "data/",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

model.save("face_mask_model.keras")

Found 6043 images belonging to 2 classes.
Found 1510 images belonging to 2 classes.
Epoch 1/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 235s 1s/step - accuracy: 0.9667 - loss: 0.0869 - val_accuracy: 0.9801 - val_loss: 0.0575
Epoch 2/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - accuracy: 0.9833 - loss: 0.0473 - val_accuracy: 0.9834 - val_loss: 0.0592
Epoch 3/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.9853 - loss: 0.0419 - val_accuracy: 0.9815 - val_loss: 0.0578
Epoch 4/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 251s 1s/step - accuracy: 0.9873 - loss: 0.0357 - val_accuracy: 0.9801 - val_loss: 0.0679
Epoch 5/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 255s 1s/step - accuracy: 0.9859 - loss: 0.0354 - val_accuracy: 0.9768 - val_loss: 0.0612
Epoch 6/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 1178s 6s/step - accuracy: 0.9878 - loss: 0.0311 - val_accuracy: 0.9821 - val_loss: 0.0586
Epoch 7/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.9902 - loss: 0.0266 - val_accuracy: 0.9881 - val_loss: 0.0453
Epoch 8/10
1

In [11]:
import cv2
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.models import load_model

yolo_model = YOLO("yolov8n-face.pt")
mask_model = load_model("face_mask_model.keras")
labels = ["No Mask", "Mask"]

def detect_and_classify(frame):
    results = yolo_model(frame)[0]
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue
        face_resized = cv2.resize(face, (224, 224)) / 255.0
        face_resized = np.expand_dims(face_resized, axis=0)
        pred = mask_model.predict(face_resized, verbose=0)[0][0]
        label = "Mask" if pred > 0.5 else "No Mask"
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    return frame